# Personalized TED Talks Recommendation System
## Milestone 1: Week 1-2 Implementation
### Module 1: Data Collection & Module 2: Data Exploration and Data Preprocessing

**Author:** Data Science Intern / SpringBoard Program  
**Goal:** Build a robust data ingestion, text preprocessing, and exploratory analysis pipeline for TED Talks recommendations.

### 1. Project Background and Module Objectives
This notebook implements the foundational data architecture for a hybrid recommender system:
- **Module 1 (Data Collection):** Collects canonical TED Talk metadata and full transcripts.
- **Module 2 (Data Exploration & Preprocessing):** Inspects completeness, resolves missing values, performs text normalization, engineers content similarity metrics, visualizes distributions, and formats collaborative user-item matrices.

In [1]:
import os
import re
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

%matplotlib inline
sns.set_theme(style='whitegrid')
print('Environment and libraries initialized successfully.')

### 2. Module 1: Data Ingestion & Audit
We load the downloaded datasets:
1. `ted_main.csv`: Metadata for 2,628 talks including titles, speakers, duration, views, tags, ratings, and events.
2. `transcripts.csv`: Verbatim transcripts for talks.

In [2]:
df_main = pd.read_csv('../data/raw/ted_main.csv', encoding='latin1')
df_trans = pd.read_csv('../data/raw/transcripts.csv', encoding_errors='replace')

print(f'Metadata shape: {df_main.shape}')
print(f'Transcripts shape: {df_trans.shape}')
df_main.head(3)

### 3. Module 2: Data Preprocessing & Merging
We perform:
- Merging on talk URL
- Imputation of missing transcripts (86 talks imputed with title + description)
- Handling missing `speaker_occupation`
- Parsing stringified Python literals (`tags`, `ratings`)

In [3]:
df_cleaned = pd.read_csv('../data/processed/ted_talks_cleaned.csv')
print('Cleaned dataset shape:', df_cleaned.shape)
print('Null values across all 43 columns:')
print(df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0])

### 4. Text Normalization Pipeline & Content Soup
To prepare for TF-IDF and embedding models in Milestone 2, we execute:
- Lowercasing
- Stripping transcript cues (e.g., `(Laughter)`, `(Applause)`)
- Removing punctuation and non-alphanumeric tokens
- Removing stopwords
- Generating an aggregated `content_soup` feature combining title, speaker, tags, occupation, and description

In [4]:
sample_idx = 0
print('--- Original Title ---')
print(df_cleaned['title'].iloc[sample_idx])
print('\n--- Original Tags ---')
print(df_cleaned['tags_list'].iloc[sample_idx])
print('\n--- Engineered Content Soup ---')
print(df_cleaned['content_soup'].iloc[sample_idx][:300] + '...')

### 5. Exploratory Data Analysis & Visualizations
Let's review the key distributions: talk duration, top topic tags, views/comments, and speaking rates.

In [5]:
plt.figure(figsize=(10, 5))
sns.histplot(df_cleaned['duration_minutes'], bins=40, kde=True, color='#e62b1e')
plt.axvline(df_cleaned['duration_minutes'].mean(), color='black', linestyle='--', label=f"Mean: {df_cleaned['duration_minutes'].mean():.1f}m")
plt.axvline(df_cleaned['duration_minutes'].median(), color='green', linestyle='-', label=f"Median: {df_cleaned['duration_minutes'].median():.1f}m")
plt.title('Talk Duration Distribution (Minutes)')
plt.legend()
plt.show()

### 6. Collaborative Filtering Setup (User-Item Interaction Matrix)
To support collaborative filtering in Milestones 2 & 3, we constructed a realistic user-item interaction benchmark spanning 500 diverse user profiles and 18,323 interactions.

In [6]:
df_interactions = pd.read_csv('../data/processed/user_item_interactions.csv')
print(f'Total interactions: {len(df_interactions):,}')
print(f'Unique users: {df_interactions["user_id"].nunique()}')
print(f'Unique talks: {df_interactions["talk_id"].nunique()}')
df_interactions.head()

### 7. Summary of Data Quality Metrics
Summary statistics capturing data richness, completeness, and distributions:

In [7]:
with open('../data/processed/data_summary_stats.json') as f:
    stats = json.load(f)
pd.Series(stats)